In [ ]:
#| default_exp probe

In [ ]:
#| export
from __future__ import annotations
import json, os, subprocess, sys, sysconfig, time
from fastcore.all import Path, first

In [ ]:
#| export
#: pywebview's GUI name per platform. A platform absent from here has no native window.
BACKENDS = {'darwin': 'cocoa', 'win32': 'edgechromium', 'linux': 'gtk'}

#: Framework builds macOS py2app can embed, best first. Not newest first: this is the version the
#: app ships, so the order is what projects are most likely to be on. uv's and pyenv's are absent
#: on purpose — both are python-build-standalone, `PYTHONFRAMEWORK` is empty, and py2app cannot
#: embed one in a bundle.
FRAMEWORK_CANDIDATES = (
    '/Library/Frameworks/Python.framework/Versions/3.13/bin/python3.13',
    '/Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14',
    '/Library/Frameworks/Python.framework/Versions/3.12/bin/python3.12',
    '/opt/homebrew/opt/python@3.13/bin/python3.13',
    '/opt/homebrew/opt/python@3.14/bin/python3.14',
    '/opt/homebrew/opt/python@3.12/bin/python3.12',
    '/usr/local/opt/python@3.13/bin/python3.13',
    '/usr/local/opt/python@3.14/bin/python3.14',
    '/usr/local/opt/python@3.12/bin/python3.12',
    '/usr/bin/python3',
)

In [ ]:
#| export
def backend(platform=None):
    "pywebview's GUI name for `platform`, or None where there is no supported webview."
    return BACKENDS.get(platform or sys.platform)

def shell_ready(platform=None):
    "`(ok, why)`: whether a native window can be opened here, and what is missing if not."
    gui = backend(platform)
    if gui is None: return False, f'no native webview backend for {platform or sys.platform}'
    try: import webview  # noqa: F401
    except ImportError as e: return False, f'pywebview is not installed ({e})'
    return True, gui

In [ ]:
#| export
def py_version(python):
    "The `(major, minor)` of `python`, or None when it will not say."
    try: out = subprocess.run([str(python), '-c', 'import sys;print(*sys.version_info[:2])'],
                              capture_output=True, text=True, timeout=30)
    except (OSError, subprocess.SubprocessError): return None
    try: return tuple(int(n) for n in out.stdout.split())
    except ValueError: return None

def is_framework(python=None):
    "Whether `python` (default: this interpreter) is a macOS framework build py2app can use."
    if python is None: return bool(sysconfig.get_config_var('PYTHONFRAMEWORK'))
    # JSON, not two words: `PYTHONFRAMEWORK` is empty off macOS, and splitting eats the version.
    probe = ('import json,sysconfig,sys;'
             "print(json.dumps([sysconfig.get_config_var('PYTHONFRAMEWORK') or '', "
             'list(sys.version_info[:2])]))')
    try: out = subprocess.run([str(python), '-c', probe], capture_output=True, text=True, timeout=30)
    except (OSError, subprocess.SubprocessError): return False
    if out.returncode: return False
    try: name, version = json.loads(out.stdout.strip() or 'null')
    except (ValueError, TypeError): return False
    return bool(name) and tuple(version) >= (3, 12)

def framework_python(candidates=FRAMEWORK_CANDIDATES):
    "The first framework interpreter on this machine that py2app can build against."
    return first(p for c in candidates if (p := Path(c)).exists() and is_framework(p))

In [ ]:
#| export
def wait_for_http(url, timeout=60, interval=.1):
    "Block until `url` answers, or `timeout` passes. True if the server came up."
    import urllib.request
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            urllib.request.urlopen(url, timeout=2)
            return True
        except Exception: time.sleep(interval)
    return False

In [ ]:
#| export
def running_from(bundle, ps_output=None):
    """The pids of processes running out of `bundle`, so a build does not overwrite a live app.

    py2app writes the bundle in place, and `python313.zip` is the running app's standard library.
    Replacing it under a live process leaves every import it has not made yet reading a file that
    is no longer the one it opened: `ZipImportError: bad local file header`, raised by whatever
    happens to import next and naming nothing that would lead anyone back to a rebuild.
    """
    target = str(Path(bundle).resolve())
    if ps_output is None:
        try: ps_output = subprocess.run(['ps', '-Ao', 'pid,command'], capture_output=True,
                                        text=True, timeout=10).stdout
        except (OSError, subprocess.SubprocessError): return []
    pids = []
    for line in ps_output.splitlines()[1:]:
        pid, _, command = line.strip().partition(' ')
        # The executable, not the whole command line: this very check names the bundle in its own
        # arguments, and so does every grep, editor and shell that has the path in it.
        exe = command.split(' ', 1)[0]
        if exe.startswith(target + os.sep) and pid.isdigit() and int(pid) != os.getpid():
            pids.append(int(pid))
    return pids